## evaluating generative text models

In [1]:
from GPTModel import GPTModel, generate_text_simple

GPT_CONFIG_124M = {
    "vocab_size": 50257,   # Vocabulary size
    "context_length": 256, # Shortened context length (orig: 1024)
    "emb_dim": 768,        # Embedding dimension
    "n_heads": 12,         # Number of attention heads
    "n_layers": 12,        # Number of layers
    "drop_rate": 0.1,      # Dropout rate
    "qkv_bias": False      # Query-key-value bias
}
model = GPTModel(GPT_CONFIG_124M)
model.eval();

In [2]:
import tiktoken
import torch

def text_to_token_ids(text, tokenizer):
    encoded = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
    encoded_tensor = torch.tensor(encoded).unsqueeze(0)
    return encoded_tensor

def token_ids_to_text(token_ids, tokenizer):
    flat = token_ids.squeeze(0)
    return tokenizer.decode(flat.tolist())

start_context = "Every effor moves you"
tokenizer = tiktoken.get_encoding("gpt2")

token_ids = generate_text_simple(
    model=model,
    idx=text_to_token_ids(start_context, tokenizer), 
    max_new_tokens=10,
    context_size=GPT_CONFIG_124M["context_length"]
)

print("output text:\n", token_ids_to_text(token_ids, tokenizer))

output text:
 Every effor moves youreat disciplinaryvg Nguyen Oliv dist SamoaKevin riversegu


## calculating the next generation loss: cross-entropy and perplexity

In [3]:
inputs = torch.tensor([[16833, 3626, 6100],   # ["every effort moves",
                       [40,    1107, 588]])   #  "I really like"]

targets = torch.tensor([[3626, 6100, 345  ],  # [" effort moves you",
                        [1107,  588, 11311]]) #  " really like chocolate"]

In [4]:
with torch.no_grad():
    logits = model(inputs)

probas = torch.softmax(logits, dim=-1)
print(probas.shape)

torch.Size([2, 3, 50257])


In [5]:
token_ids = torch.argmax(probas, dim=-1, keepdim=True)
print("token IDs:\n", token_ids)

token IDs:
 tensor([[[ 3332],
         [17167],
         [  975]],

        [[ 9867],
         [41674],
         [  402]]])


In [6]:
tokenizer = tiktoken.get_encoding("gpt2")
print(f"targets batch 1: {token_ids_to_text(targets[0], tokenizer)}")
print(f"outputs batch 1: {token_ids_to_text(token_ids[0].flatten(), tokenizer)}")

targets batch 1:  effort moves you
outputs batch 1:  satetaween


## cross-entropy loss

<div class="alert alert-block alert-success">

The token probabilities corresponding to the target indices are as follows:


</div>

In [7]:
text_idx = 0
target_probas_1 = probas[text_idx, [0, 1, 2], targets[text_idx]]
print("text 1:", target_probas_1)

text_idx = 1
target_probas_2 = probas[text_idx, [0, 1, 2], targets[text_idx]]
print("text 2:", target_probas_2)

text 1: tensor([8.1508e-06, 1.3751e-05, 2.2203e-05])
text 2: tensor([2.2053e-05, 1.6521e-05, 2.0018e-05])


In [8]:
log_probas = torch.log(torch.cat((target_probas_1, target_probas_2)))
print(log_probas)

tensor([-11.7174, -11.1944, -10.7153, -10.7220, -11.0109, -10.8189])


In [9]:
avg_log_probas = torch.mean(log_probas)
print(avg_log_probas)

tensor(-11.0298)


In [10]:
neg_avg_log_probas = avg_log_probas * -1
print(neg_avg_log_probas)

tensor(11.0298)


In [11]:
print("logits shape:", logits.shape)
print("targets shape:", targets.shape)

logits shape: torch.Size([2, 3, 50257])
targets shape: torch.Size([2, 3])


In [12]:
logits_flat = logits.flatten(0, 1)
targets_flat = targets.flatten()

print("flattened logits:", logits_flat.shape)
print("flattened targets:", targets_flat.shape)

flattened logits: torch.Size([6, 50257])
flattened targets: torch.Size([6])


In [14]:
loss = torch.nn.functional.cross_entropy(logits_flat, targets_flat)
print(loss)

tensor(11.0298)


## Perplexity

In [16]:
perplexity = torch.exp(loss)
print(perplexity)

tensor(61685.7969)
